# Statistics

## Load Model

In [ ]:
from wings.modeling.loss import DiceLoss, WeightedDiceLoss
from wings.modeling.litnet import LitNet
from wings.config import PROCESSED_DATA_DIR, MODELS_DIR
from wings.dataset import MaskRectangleDataset
from wings.modeling.unet import UNet
import torch
from ultralytics import YOLO

mean_coords = torch.load(
    PROCESSED_DATA_DIR / "mask_datasets" / 'rectangle' / "mean_shape.pth", weights_only=False
)

# checkpoint_path = MODELS_DIR / "new_unet" / "custom-unet-pretrained-epoch=49-val_loss=0.02-custom-unet-training_1.ckpt"
checkpoint_path = MODELS_DIR / "new_unet" / "last.ckpt" # ten last to nie jest prawdziwy last tylko jakis z 54 epoki

unet_model = UNet(in_channels=1, out_channels=1, kernel_size=3)
model = LitNet.load_from_checkpoint(checkpoint_path, model=unet_model, criterion=WeightedDiceLoss(landmark_weight=50.0, background_weight=1.0))
model.eval()


yolo_model = YOLO(MODELS_DIR / "yolo26n" / "best.pt").to("cuda")
# yolo_model = YOLO(MODELS_DIR / "best.pt").to("cuda")


2026-05-10 19:51:21.718 | INFO     | wings.config:<module>:44 - PROJ_ROOT path is: C:\Users\X\projects\bees
2026-05-10 19:51:23.568 | INFO     | wings.config:<module>:68 - torch.cuda.get_device_name()='NVIDIA RTX A3000 12GB Laptop GPU'
W0510 19:51:29.290000 6404 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# Test dataset

In [3]:
test_dataset = torch.load(
    PROCESSED_DATA_DIR / "mask_datasets" / 'rectangle-cropped' / "test_mask_dataset_ch1_400.pth",
    weights_only=False
)
max_n = len(test_dataset)
max_n


2172

In [4]:
from wings.gpa import recover_order, center_shape, normalize_shape, procrustes_align
from wings.visualizing.image_preprocess import final_coords
from wings.gpa import handle_coordinates
import torch
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

original_labels = [[] for _ in range(5)]
predicted_labels = [[] for _ in range(5)]
points_indices = [[] for _ in range(5)]

for idx, (image, _, coords, (x_size, y_size)) in enumerate(tqdm(test_dataset, desc="Evaluating")):
    image = image.to(device).unsqueeze(0)
    output = model(image)
    mask = torch.round(output).squeeze().detach().cpu().numpy()
    mask_coords = final_coords(mask, x_size, y_size)
    mask_coords = torch.tensor(mask_coords)
    n_points = len(mask_coords)

    if n_points < 18:
        idx_group = 0
    elif n_points == 18:
        idx_group = 1
    elif n_points == 19:
        idx_group = 2
    elif n_points == 20:
        idx_group = 3
    else:
        idx_group = 4

    reordered = handle_coordinates(mask_coords, mean_coords).cpu().numpy()
    orig = coords.view(-1, 2)

    points_indices[idx_group].append(idx)
    original_labels[idx_group].append(orig.cpu().numpy())
    predicted_labels[idx_group].append(reordered)



original_labels = [np.stack(lbls) if len(lbls) > 0 else np.empty((0, 2)) for lbls in original_labels]
predicted_labels = [
    np.stack(lbls) if len(lbls) > 0 else np.empty((0, 0, 2)) for lbls in predicted_labels
]


total_samples_num = len(test_dataset)
failed_samples_num = sum(len(points_indices[i]) for i in [0, 1, 3, 4])  # exclude 19-points group

print(f"Total samples: {total_samples_num}")
print(f"Failed masks: {failed_samples_num}")

for i, n_points in enumerate(range(17, 22)):
    print(f"predicted_labels_{n_points}.shape = {predicted_labels[i].shape}")



errors = []
for i in range(5):
    if len(predicted_labels[i]) == 0 or len(original_labels[i]) == 0:
        errors.append(np.empty((0, 0)))
        continue
    err = np.linalg.norm(predicted_labels[i] - original_labels[i], axis=2)
    errors.append(err)

# Compute means and medians
for i, n_points in enumerate(range(17, 22)):
    if errors[i].size == 0:
        print(f"{n_points}-point group: no samples available.")
        continue
    mean_val = errors[i].mean()
    median_val = np.median(errors[i])
    num_img = len(errors[i])  # total number of point-wise errors

    print(f"mean{n_points}={mean_val:.4f}\tmedian{n_points}={median_val:.4f}\t\timages={num_img}")



# ---- GLOBAL ERROR ACROSS ALL GROUPS ----
# Flatten all non-empty error arrays into a single array
all_errors = np.concatenate([e.flatten() for e in errors if e.size > 0])

if all_errors.size > 0:
    global_mean = all_errors.mean()
    global_median = np.median(all_errors)
    print(f"\nGLOBAL MEAN ERROR = {global_mean:.4f}")
    print(f"GLOBAL MEDIAN ERROR = {global_median:.4f}")
else:
    print("\nNo valid error data available for global computation.")


Evaluating: 100%|██████████| 2172/2172 [00:53<00:00, 40.32it/s]

Total samples: 2172
Failed masks: 35
predicted_labels_17.shape = (0, 0, 2)
predicted_labels_18.shape = (6, 19, 2)
predicted_labels_19.shape = (2137, 19, 2)
predicted_labels_20.shape = (28, 19, 2)
predicted_labels_21.shape = (1, 19, 2)
17-point group: no samples available.
mean18=1.5711	median18=1.2760		images=6
mean19=1.3260	median19=1.2228		images=2137
mean20=1.3571	median20=1.2020		images=28
mean21=1.6896	median21=1.3004		images=1

GLOBAL MEAN ERROR = 1.3273
GLOBAL MEDIAN ERROR = 1.2227


# Puławy

In [5]:
from wings.config import RAW_DATA_DIR
import random
import os
from PIL import Image
from pathlib import Path


directory = RAW_DATA_DIR / "pulawy" / "01_00_01"

all_files = []
for subdir, _, files in os.walk(directory):
    for file in files:
        if file.lower().endswith(".png"):
            path = Path(os.path.join(subdir, file))

            with Image.open(path) as img:
                x_size, y_size = img.size

            if x_size == 5782:
                continue

            all_files.append(path)


sample_path = random.choice(all_files)
sample_path

WindowsPath('C:/Users/X/projects/bees/data/raw/pulawy/01_00_01/01_00_19/01_00_515.png')

In [6]:
from PIL import Image
from wings.gpa import handle_coordinates
from wings.visualizing.image_preprocess import final_coords, unet_fit_rectangle_preprocess
import torch
import numpy as np
from tqdm import tqdm
import cv2
from functools import partial

original_labels = [[] for _ in range(5)]
predicted_labels = [[] for _ in range(5)]
points_indices = [[] for _ in range(5)]
negative_coords = []

preprocess = partial(unet_fit_rectangle_preprocess, output_size=400)

for idx, img_path in enumerate(tqdm(all_files, desc="Processing images")):
    img = Image.open(img_path)
    img.load()
    meta = img.info['IdentiFly']
    orig = meta.split("landmarks:")[1].split(";")[0]
    labels = np.array([int(x) for x in orig.split()])
    x_coords, y_coords = labels[::2], labels[1::2]
    img = cv2.imread(str(img_path))
    result = yolo_model.predict(img, verbose=False)[0]
    if len(result) > 0:
        boxes = result.boxes.cpu().numpy()
        conf = boxes.conf
        x1, y1, x2, y2 = boxes.xyxy[np.argmax(conf)]

        xmin, xmax = int(min(x1, x2)), int(max(x1, x2))
        ymin, ymax = int(min(y1, y2)), int(max(y1, y2))

        img = img[ymin:ymax, xmin:xmax]
        x_coords = x_coords - xmin
        y_coords = y_coords - ymin

    if (x_coords < 0).any() or (y_coords < 0).any():
        negative_coords.append(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_tensor = torch.from_numpy(img).unsqueeze(0).contiguous()
    x_size, y_size = img_tensor.shape[2], img_tensor.shape[1]
    image_tensor, _, _ = preprocess(img_tensor)

    output = model(image_tensor.cuda().unsqueeze(0))
    mask = torch.round(output).squeeze().detach().cpu().numpy()
    mask_coords = final_coords(mask, x_size, y_size)
    mask_coords = torch.tensor(mask_coords)
    n_points = len(mask_coords)

    if n_points < 18:
        idx_group = 0
    elif n_points == 18:
        idx_group = 1
    elif n_points == 19:
        idx_group = 2
    elif n_points == 20:
        idx_group = 3
    else:
        idx_group = 4

    reordered = handle_coordinates(mask_coords, mean_coords).cpu().numpy()
    x_coords = torch.tensor(x_coords)
    y_coords = torch.tensor(y_coords)
    y_coords = y_size - y_coords - 1
    orig = torch.stack((x_coords, y_coords), dim=1)


    points_indices[idx_group].append(img_path)
    original_labels[idx_group].append(orig.cpu().numpy())
    predicted_labels[idx_group].append(reordered)



original_labels = [np.stack(lbls) if len(lbls) > 0 else np.empty((0, 2)) for lbls in original_labels]
predicted_labels = [
    np.stack(lbls) if len(lbls) > 0 else np.empty((0, 0, 2)) for lbls in predicted_labels
]


total_samples_num = len(all_files)
failed_samples_num = sum(len(points_indices[i]) for i in [0, 1, 3, 4])  # exclude 19-points group

print(f"Total samples: {total_samples_num}")
print(f"Failed masks: {failed_samples_num}")

for i, n_points in enumerate(range(17, 22)):
    print(f"predicted_labels_{n_points}.shape = {predicted_labels[i].shape}")


print(f"{len(negative_coords)=}")


errors = []
for i in range(5):
    if len(predicted_labels[i]) == 0 or len(original_labels[i]) == 0:
        errors.append(np.empty((0, 0)))
        continue
    err = np.linalg.norm(predicted_labels[i] - original_labels[i], axis=2)
    errors.append(err)

# Compute means and medians
for i, n_points in enumerate(range(17, 22)):
    if errors[i].size == 0:
        print(f"{n_points}-point group: no samples available.")
        continue
    mean_val = errors[i].mean()
    median_val = np.median(errors[i])
    num_img = len(errors[i])  # total number of point-wise errors

    print(f"mean{n_points}={mean_val:.4f}\tmedian{n_points}={median_val:.4f}\t\timages={num_img}")



# ---- GLOBAL ERROR ACROSS ALL GROUPS ----
# Flatten all non-empty error arrays into a single array
all_errors = np.concatenate([e.flatten() for e in errors if e.size > 0])

if all_errors.size > 0:
    global_mean = all_errors.mean()
    global_median = np.median(all_errors)
    print(f"\nGLOBAL MEAN ERROR = {global_mean:.4f}")
    print(f"GLOBAL MEDIAN ERROR = {global_median:.4f}")
else:
    print("\nNo valid error data available for global computation.")

Processing images: 100%|██████████| 1012/1012 [00:57<00:00, 17.49it/s]

Total samples: 1012
Failed masks: 112
predicted_labels_17.shape = (0, 0, 2)
predicted_labels_18.shape = (5, 19, 2)
predicted_labels_19.shape = (900, 19, 2)
predicted_labels_20.shape = (95, 19, 2)
predicted_labels_21.shape = (12, 19, 2)
len(negative_coords)=0
17-point group: no samples available.
mean18=2.9509	median18=2.2332		images=5
mean19=2.5440	median19=1.8916		images=900
mean20=2.6694	median20=1.9300		images=95
mean21=2.5429	median21=1.8949		images=12

GLOBAL MEAN ERROR = 2.5578
GLOBAL MEDIAN ERROR = 1.8954
